## Observations

**1. Both schemes are essentially free at low load factors.** Up through $\alpha$ ≈ 0.25 both modes sit very near 1.0–1.4 probes per insertion. There is no meaningful performance difference between linear and quadratic at low $\alpha$ — if you tested small tables you may conclude the two schemes are equivalent, which is the wrong conclusion and a good teaching moment.

**2. Linear and quadratic diverge sharply once $\alpha$ crosses ≈ 0.5.** Below $\alpha = 0.5$ the two curves are indistinguishable (and actually trade places due to noise — linear is sometimes slightly cheaper). Above 0.5, linear's probe count explodes while quadratic grows much more slowly. At $\alpha = 0.79$, linear already averages about 7.7 probes per insertion; quadratic is still under 5. At $\alpha = 0.94$, linear costs about 27.7 probes while quadratic costs about 13 — roughly a 2× gap that keeps widening.

**3. Primary clustering is visible in the linear numbers.** The jump from ~2 probes at $\alpha = 0.5$ to ~52 probes at $\alpha = 1.0$ is far steeper than linear growth — it's the quadratic-in-$(1/(1−\alpha))$ blowup predicted by the formula in the handout. The measured linear values match $\tfrac{1}{2}(1 + 1/(1-\alpha)^2)$ well up through $\alpha ≈ 0.5$, then fall below theory at the high end. That gap is a finite-capacity artifact: in a 101-slot table the variance is bounded by the table size, so we cannot observe the asymptotic pile-ups the formula predicts for an infinite table.

**4. Quadratic probing is the one that actually fails, and it fails well before the table is full.** Linear probing recorded **zero** failed attempts across 500 trials — confirming the guarantee that linear probing finds an empty slot whenever one exists. Quadratic probing racked up **747** failed attempts. Those failures are concentrated at the very top of the load-factor range: once $\alpha$ gets into the 0.9+ region, individual random strings start hashing to slots whose quadratic probe sequence cycles through only occupied positions before exhausting all 101 attempts. The simulation retries with fresh strings and eventually succeeds, but the failure count is the experimental signature of quadratic probing's coverage limitation. Note that even though the table size is prime (so the $\alpha < 0.5$ theorem applies), the guarantee stops at $\alpha = 0.5$ — above that there is no guarantee, and these failures are exactly what the theorem warns about.

**5. Practical takeaway that falls out of the data.** The commonly cited rules of thumb — resize linear-probing tables around $\alpha$ ≈ 0.7 and quadratic tables around $\alpha$ ≈ 0.5 — look conservative from this data, but only if you're comfortable with average probe counts of 5+. If you want single-digit probe counts with high confidence, the 0.5–0.7 resize window is exactly right: that is the last region where both curves are still growing gently.

In [ ]:
"""
Week 11 — Measuring probes per insertion vs. load factor
for linear and quadratic probing in an open-addressed hash table.

Failure policy (Option A from the handout):
  - Only successful insertions contribute to the averages.
  - Failed attempts are counted separately and reported at the end.
  - On failure we discard the string and try another, so the simulation
    keeps trying to reach higher load factors.
"""

from __future__ import annotations

import random
import string


# =========================================================================
# ProbingHashTable (from the assignment, unmodified)
# =========================================================================

class ProbingHashTable:
    _DEFAULT_CAPACITY: int = 10
    _DEFAULT_PROBING_MODE: str = "linear"
    _PROBE_MODES: tuple[str, str] = ("linear", "quadratic")
    _ERROR_FULL: str = "Hash table is full"

    def __init__(self, capacity: int = _DEFAULT_CAPACITY,
                 mode: str = _DEFAULT_PROBING_MODE):
        if mode not in self._PROBE_MODES:
            mode = self._DEFAULT_PROBING_MODE
        self._capacity = capacity
        self._mode = mode
        self._table: list[str | None] = [None] * capacity
        self._size = 0

    def _hash(self, key: int) -> int:
        return key % self._capacity

    def _probe(self, position: int, attempt: int) -> int:
        if self._mode == self._DEFAULT_PROBING_MODE:  # linear
            return (position + attempt) % self._capacity
        else:  # quadratic
            return (position + attempt * attempt) % self._capacity

    def load_factor(self) -> float:
        return self._size / self._capacity

    def insert(self, value: str) -> list[int]:
        probes: list[int] = []
        if self._size < self._capacity:
            key: int = abs(hash(value))
            index: int = self._hash(key)
            i: int = 0
            insertion_successful: bool = False
            while i < self._capacity and not insertion_successful:
                probe_index: int = self._probe(index, i)
                probes.append(probe_index)
                slot: str | None = self._table[probe_index]
                insertion_successful = slot is None or slot == value
                if insertion_successful:
                    self._table[probe_index] = value
                    if slot is None:
                        self._size += 1
                i += 1
        return probes


# =========================================================================
# Helper: generate a random alphabetic string
# =========================================================================

def random_string(n: int) -> str:
    """Generate a random string of length n using alphabetical characters."""
    return "".join(random.choices(string.ascii_letters, k=n))


# =========================================================================
# The experiment 
# =========================================================================

def main() -> None:

    # -------------------------------------------------------------------
    # Configuration
    # -------------------------------------------------------------------
    # CAPACITY is prime, as recommended by the handout. Quadratic probing's
    # alpha < 0.5 coverage guarantee requires a prime table size.
    # TRIALS is the number of fresh tables we will fill per mode. More
    # trials smooths out the noise in the averages.
    # STRING_LEN is long enough that accidental duplicate strings are
    # vanishingly unlikely.

    CAPACITY: int = 101
    TRIALS: int = 500
    STRING_LEN: int = 8

    # A safety cap on how many insert attempts we make per trial. A healthy
    # trial finishes in about CAPACITY attempts. Quadratic probing can stall
    # near full, so we need a ceiling. 20 * CAPACITY is far more attempts
    # than any healthy trial ever needs.
    MAX_ATTEMPTS: int = 20 * CAPACITY

    random.seed(42)  # make the experiment reproducible

    # -------------------------------------------------------------------
    # Storage for measurements
    # -------------------------------------------------------------------
    # For each mode we keep two parallel lists of length CAPACITY:
    #   sum_probes[k]   = total probes observed for the (k+1)-th successful
    #                     insertion, summed over all trials
    #   count_probes[k] = number of trials that produced a (k+1)-th
    #                     successful insertion
    # The load factor for index k is (k+1) / CAPACITY. The average probes
    # at that load factor is sum_probes[k] / count_probes[k].

    linear_sum_probes: list[int] = [0] * CAPACITY
    linear_count_probes: list[int] = [0] * CAPACITY
    linear_failures: int = 0

    quad_sum_probes: list[int] = [0] * CAPACITY
    quad_count_probes: list[int] = [0] * CAPACITY
    quad_failures: int = 0

    # -------------------------------------------------------------------
    # Run the LINEAR probing experiment
    # -------------------------------------------------------------------
    # For each of TRIALS trials: build a fresh table, then keep inserting
    # random strings until the table is full (or we hit the safety cap).
    # Record probe counts per load-factor step, and count failed attempts.

    trial = 0
    while trial < TRIALS:
        table = ProbingHashTable(capacity=CAPACITY, mode="linear")
        attempts = 0
        while table._size < CAPACITY and attempts < MAX_ATTEMPTS:
            size_before = table._size
            probes = table.insert(random_string(STRING_LEN))
            if table._size > size_before:
                # Successful insertion. The new size tells us which
                # load-factor bucket this belongs to.
                bucket = table._size - 1
                linear_sum_probes[bucket] += len(probes)
                linear_count_probes[bucket] += 1
            else:
                # Failed insertion: load factor did not advance.
                # Discard this attempt and try a different string.
                linear_failures += 1
            attempts += 1
        trial += 1

    # -------------------------------------------------------------------
    # Run the QUADRATIC probing experiment
    # -------------------------------------------------------------------
    # Same structure, different mode. Kept as a separate block rather than
    # wrapped in a loop over modes so the logic stays linear and readable.

    trial = 0
    while trial < TRIALS:
        table = ProbingHashTable(capacity=CAPACITY, mode="quadratic")
        attempts = 0
        while table._size < CAPACITY and attempts < MAX_ATTEMPTS:
            size_before = table._size
            probes = table.insert(random_string(STRING_LEN))
            if table._size > size_before:
                bucket = table._size - 1
                quad_sum_probes[bucket] += len(probes)
                quad_count_probes[bucket] += 1
            else:
                quad_failures += 1
            attempts += 1
        trial += 1

    # -------------------------------------------------------------------
    # Print the results
    # -------------------------------------------------------------------
    # Columns: load factor, average probes (linear), average probes (quadratic)
    # A dash means no trial reached that load factor (quadratic only).

    print("load  linear  quadratic")
    k = 0
    while k < CAPACITY:
        load = (k + 1) / CAPACITY

        if linear_count_probes[k] > 0:
            lin = linear_sum_probes[k] / linear_count_probes[k]
            lin_text = f"{lin:.2f}"
        else:
            lin_text = "-"

        if quad_count_probes[k] > 0:
            quad = quad_sum_probes[k] / quad_count_probes[k]
            quad_text = f"{quad:.2f}"
        else:
            quad_text = "-"

        print(f"{load:.2f}  {lin_text}  {quad_text}")
        k += 1

    print()
    print(f"Linear failures: {linear_failures}")
    print(f"Quadratic failures: {quad_failures}")



if __name__ == "__main__":
    main()

load  linear  quadratic
0.01  1.00  1.00
0.02  1.01  1.01
0.03  1.02  1.03
0.04  1.03  1.03
0.05  1.04  1.05
0.06  1.05  1.05
0.07  1.06  1.06
0.08  1.09  1.08
0.09  1.08  1.08
0.10  1.09  1.12
0.11  1.09  1.11
0.12  1.14  1.15
0.13  1.15  1.16
0.14  1.12  1.13
0.15  1.16  1.20
0.16  1.19  1.18
0.17  1.20  1.21
0.18  1.18  1.22
0.19  1.27  1.24
0.20  1.26  1.26
0.21  1.26  1.30
0.22  1.28  1.25
0.23  1.30  1.28
0.24  1.35  1.27
0.25  1.31  1.36
0.26  1.33  1.39
0.27  1.42  1.40
0.28  1.43  1.37
0.29  1.44  1.40
0.30  1.44  1.44
0.31  1.52  1.38
0.32  1.59  1.52
0.33  1.50  1.48
0.34  1.53  1.51
0.35  1.58  1.53
0.36  1.70  1.52
0.37  1.55  1.64
0.38  1.80  1.66
0.39  1.78  1.66
0.40  1.76  1.65
0.41  1.82  1.75
0.42  1.86  1.74
0.43  1.97  1.82
0.44  2.03  1.79
0.45  2.06  1.82
0.46  2.03  2.00
0.47  2.10  1.97
0.48  2.18  2.00
0.49  2.29  2.03
0.50  2.28  1.91
0.50  2.41  2.21
0.51  2.31  2.09
0.52  2.54  2.16
0.53  2.66  2.27
0.54  2.45  2.24
0.55  2.66  2.25
0.56  2.72  2.44
0.57  3